# Final Data Preparation (Datasets merging, future engeneering and modeling preparation)

#### Currently the project data is seperated into miltiple datasets, each with its own specific data. What this notebook will do, is that it will merge the datasets by structuring them into ones, with organaized and detailly selected features.The main aim of the notebook is to prepare the data for analyses and more importantly for modeling! 

The following components will form the Final dataset: \
All of the three data sources will be used for the creation of the final dataset: **Football-Data.co.uk, EloRatings and Understat data**.

## As features it wil contain:

### 1.Base Matches data (Match-level)

##### Base volume data 
**"season", "datetime", "h_title", "a_title"** \
**"home_goals_full", "away_goals_full", "result_full"** \
**"home_shots", "away_shots", "home_shots_on_target", "away_shots_on_target"** \
**"home_fouls", "away_fouls"**

##### Odds
**"odds_bet365_home", "odds_bet365_draw", "odds_bet365_away"**

##### Elo ratings
**"home_elo", "away_elo", "elo_diff"**

##### Expexted goals
**"xG_h", "xG_a"**

##### Rolling features(Rolling form features are among the strongest predictors!): 
**home_rolling_xG_last_5, away_rolling_xG_last_5** \
**home_rolling_xGA_last_5, away_rolling_xGA_last_5**

##### Form(Points gathred from matches)
**home_form_last_5, away_form_last_5** \
**rolling_shots_last_5, rolling_sot_last_5**

---

### 2.Shot Aggregation data (Match-level)
Matches shot-level data

#### As features it will contain:

**"home_big_chances", "away_big_chances"** \
**"home_avg_xG_per_shot", "away_avg_xG_per_shot"**

---

# Final Dataset(Match-level):
The final dataset will be one perfectly structured, organized, cleaned and prepared dataset which will contain data at match level!As the purpose of the statistcial models is to estimate probabilities for outcomes of matches, it is logical for them to work with matches data!

#### Here is what the final dataset will, eventually contains:

##### AS IDENTIFIERS:
- season
- datetime
- h_title
- a_title

##### AS TOTAL STATS:
- home_goals_full
- away_goals_full
- home_shots_on_target
- away_shots_on_target
- home_fouls
- away_fouls
- result_full

##### Betting odds:
- odds_bet365_home
- odds_bet365_draw
- odds_bet365_away
- handicap_home
- handicap_away

##### Elo ratings:
- elo_difference
- home_elo
- away_elo

##### Rolling Forms:
- home_rolling_xG_5 (Aggregated expected goals last five matches)
- away_rolling_xG_5

- home_rolling_xGA_5 (Aggregated expected goals againts last five matches)
- away_rolling_xGA_5

- home_form_points_5 (Aggregated points won last five matches)
- away_form_points_5

- home_rolling_shots_last_5 (Aggregated shots last five matches)
- away_rolling_shots_last_5
- home_rolling_sot_last_5 (Aggregated shots on target last five matches)
- away_rolling_sot_last_5

##### Team Strength
- home_attack_xG_total
- away_attack_xG_total

- home_defense_xG_total
- away_defense_xG_total

##### Metrics differences:
- xG_diff
- goal_diff
- rolling_goal_diff
- rolling_xG_diff
- h2h_record

### This is what the final dataset should look like, plus-minus some additional features, which could be added in time, only if needed!The data in the final dataset is specifically and detaily selected for its influential factors and it is known to be very efficient for modeling and matches predictions!

Now, as the datasets strcutures are explained, lets start!

### Loading the required libraries:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from football_betting_analysis.config import PROCESSED_DATA_PATH
from football_betting_analysis.data.data_cleaning import optimize_dataframe_memory
from football_betting_analysis.data.data_cleaning import validate_and_cast_dataframe_dtypes
from football_betting_analysis.data.match_resolver import resolve_date_shift_matches
from football_betting_analysis.data.save_data_into_file import save_data

### Loading the datasets:

In [2]:
# Football-Data.co.uk data
football_data_df = pd.read_parquet('../../data/interim/football_data_co_uk/interim_matches_v1.parquet')

# Elo ratings data
elo_ratings_df = pd.read_parquet('../../data/interim/elo_ratings/interim_elo_ratings_v1.parquet')
elo_ratings_matches_df = pd.read_parquet('../../data/interim/elo_ratings/interim_elo_matches_v1.parquet')

# Understat data
understat_matches_df = pd.read_parquet('../../data/interim/understat_data/interim_matches_v1.parquet')
understat_matches_shots_df = pd.read_parquet('../../data/interim/understat_data/interim_matches_shots_v1.parquet')
understat_players_df = pd.read_parquet('../../data/interim/understat_data/interim_players_v1.parquet')
understat_teams_context_df = pd.read_parquet('../../data/interim/understat_data/interim_teams_context_v1.parquet')

### Prerequestins:

#### Storing datasets:
I want to specify that these datasets will be stored into the `data/processed` directory, which directoty is for storing finaly processed and totally cleaned datasets which are ready for analyses and modeling!This will also be the place where the final dataset will be stored!

---

### Initial cleaning

#### Merging requirements:
Before merging the datasets we should confirm three very important detailes: \
1.**STANDARDIZED COLUMN NAMES**:
- The datasets should have a unique key by which they to be merged!As the datasets are not collected from one source, they do not have predefined matches ids, which means that this key should be constructed from the features in the datasets.This unique key surely will be the combination between: **season, datetime, h_title, a_title** - In the cleaning phase, we have guaranteed that there is no a dataset which violates this unique criteria.Also the datasets are all designed to have these unique features named in a consistent way, which means that all of the datasets have them and we are freely to use them as a unique key!

2.**STANDARDIZED TEAM NAMES**:
- The next thing which is extrmely important and required for the datasets to be merged correctly, is for them to have consistent team names.This also is something which is already achived in the cleaning phase and currently all og the datasets have valid and consistent teams names

3.**STANDARDIZED DATETIME**:
- Another thing, is that the datasets should have their datetimes in one format, which should be the standard one: `YYYYYYYY-MM-DD` without any timestamps etc.This is something which is valid for the Elo Ratigns dataset and for the Football-Data.co.uk dataset, but the Understat dataset contains its datetimes with timestamps.So this should be fixed!

So, the only thing that should be done is to normalize the datetime column of the Understats datasets.Dataset**s** because this should also be done for the Understat matches shots dataset!

So lets do this for both Understat matches and matches shots datasets!

In [3]:
understat_matches_df['datetime'] = understat_matches_df['datetime'].dt.normalize()
understat_matches_shots_df['datetime'] = understat_matches_shots_df['datetime'].dt.normalize()

Ok now everything is perfect!

---

### Creating the Base Match dataset:

I will start by constructing the first dataset, which is the main matches dataset.This dataset should contain the main match-level data and its data will be mainly used for the Final dataset.

To do this I should use all of the three datasets.

- I will get the main match stats from the `Elo_Ratings matches dataset`, which dataset will also give me the elo ratings metrics which are important.

- I will use the Football-Data.co.uk dataset to get the matches betting odds, which in fact are also contained into the Elo Ratings datasets, but the Footbal-Data.co.uk is much more reliable and specifically created to provide betting odds!

- And last but not least, I will use the Understat dataset which will give me the **expected goals** metrics, which are already known to be extremely important!

So lets begin.

In [4]:
matches_final_df = pd.merge(
    left=elo_ratings_matches_df, 
    right=understat_matches_df,
    on=['season', 'datetime', 'h_title', 'a_title'], 
    how='inner',
    indicator=True
)

Now lets see if everything has matched, by simply checking if the length of the dataset is equal to 4180, which is the total amoung of matches for all of the matches datasets!

In [5]:
len(matches_final_df) == 4180

False

In [6]:
len(matches_final_df)

4111

Well, we can see that there is some problem with the matching, because there are **69** matches which does not match the unique criteria!

Lets see from where this might be coming!Where the matches differ? \
I will see that by using an outer join.

In [7]:
elo_ratings_matches_df = elo_ratings_matches_df.copy()
understat_matches_df = understat_matches_df.copy()

# Creating helpers
# I am doing that because, in order to fix the problem with the dates, I need to store the indeces of datasets
# so that later I can use them to locate the unmatching matches and fix them!
elo_ratings_matches_df["elo_rating_idx"] = elo_ratings_matches_df.index
understat_matches_df["understat_idx"] = understat_matches_df.index

In [8]:
exact_merge = elo_ratings_matches_df.merge(
    understat_matches_df,
    on=['season', 'datetime', 'h_title', 'a_title'], 
    how='outer',
    indicator=True,
    suffixes=("", "_u"),
)

Now lets take only the ones which didn't match:

In [9]:
left_only = exact_merge[exact_merge["_merge"] == "left_only"].copy()
right_only = exact_merge[exact_merge["_merge"] == "right_only"].copy()

#### As I have inspected the unmatching matches, I have identified the problem, which is simply caused from the the **discrepancies between the dates** of the different datasets.And more specifically the problem comes from the fact that certain matches from the **Understat data** are with incorrectly defined dates!This could have happen from problems while the scraping, or just inconsistencies between the time zones and many more other reasons.This is not important.What is important is that the problem is identified and it should be fixed.

So what I will do is that I will use an automated tolerant matching function, which will take the two unmatching extracts from the data frames, and will try to match the football games with small date inconsistencies - max 1 day, because otherwise the matches could be totally different, but it is just not possible, in a domestic league like La Liga, two teams to have played two matches two days in row - It is just not possible!

The function which I am going to use is defined at: [resolve_date_shift_matches](../../src/football_betting_analysis/data/match_resolver.py)

So lets apply the function:

In [10]:
resolved_matches = resolve_date_shift_matches(
    left_df=left_only,
    right_df=right_only,
    use_index=True,
    left_index_col='elo_rating_idx',
    right_index_col='understat_idx',
    max_days=1
)

In [11]:
resolved_matches

,left_index,right_index
0,114738.0,1403.0
1,114894.0,1406.0
2,114893.0,1407.0
3,115075.0,1412.0
4,115076.0,1413.0
...,...,...
64,140019.0,7915.0
65,141805.0,7967.0
66,142388.0,7987.0
67,196423.0,17503.0


Now as we have collected the unmatching matches in pairs, lets fix the Undrstat ones, by simply updating the date of the matches one day before!

In [12]:
for _, row in resolved_matches.iterrows():

    left_idx = row["left_index"]
    right_idx = row["right_index"]

    corrected_date = elo_ratings_matches_df.loc[left_idx, "datetime"]

    understat_matches_df.loc[right_idx, "datetime"] = corrected_date

And now lets remove the helping columns from the datasets!

In [13]:
elo_ratings_matches_df = elo_ratings_matches_df.drop(columns=['elo_rating_idx'])
understat_matches_df = understat_matches_df.drop(columns=['understat_idx'])

And now lets again make the merge, and see if we have fixed the problems with the dates:

In [14]:
test_df = pd.merge(
    left=elo_ratings_matches_df, 
    right=understat_matches_df,
    on=['season', 'datetime', 'h_title', 'a_title'], 
    how='inner'
)

In [15]:
len(test_df)

4180

Yes everything is perfect!We are good to go!

Now before moving to the merges, I will define the base dataset which serve as a starting point to the merges.This base dataset will be the Elo Ratings dataset, because it contain almost everything that the **Base Match**(The one that we are currently creating) dataset will need and the other things will be gathered from the other two datasets(The betting odds and the expected goals).

In [16]:
base_matches_df = elo_ratings_matches_df.copy()

Now what I will do is that I will merge this dataset with **Understat** by selecting only the **specific columns** that I want to get from the Understat matches data, just because the only thing that I need from the understat data are the **expected goals metrics**, which I do not have in any other dataset!

So here is what I will get from understat:

In [17]:
UNDERSTAT_MATCH_FEATURES = [
    "season",
    "datetime",
    "h_title",
    "a_title",
    "xG_h",
    "xG_a",
]

In [18]:
base_matches_df = base_matches_df.merge(
    understat_matches_df[UNDERSTAT_MATCH_FEATURES],
    on=["season", "datetime", "h_title", "a_title"],
    how="left",
    validate="one_to_one"
)

### Checking for some duplicates and missing values:

In [19]:
base_matches_df[['xG_h', 'xG_a']].isna().any()

xG_h    False
xG_a    False
dtype: bool

In [20]:
assert (
    base_matches_df[
        ["season", "datetime", "h_title", "a_title"]
    ]
    .duplicated()
    .sum()
    == 0
)

#### Now lets also merge the Football-Data.co.uk dataset too!

From Football-Data.co.uk I will get only the odds metrics, so:

In [21]:
FOOTBALL_DATA_FEATURES = [
    "season",
    "datetime",
    "h_title",
    "a_title",
    "odds_bet365_home",
    "odds_bet365_draw",
    "odds_bet365_away"
]

In [22]:
base_matches_df = base_matches_df.merge(
    football_data_df[FOOTBALL_DATA_FEATURES],
    on=['season', 'datetime', 'h_title', 'a_title'],
    how='left',
    validate='one_to_one',
    suffixes=['_elo', '']
)

Now lets drop the **EloRatings** odds columns, and leave only the ones from the **Football-Data**.

In [23]:
base_matches_df = base_matches_df.drop(
    columns=[
        "odds_bet365_home_elo",
        "odds_bet365_draw_elo",
        "odds_bet365_away_elo"
    ]
)

## Removing features

Removing columns which are totally useless and will not take any role into the analyses or into the modeling!

In [24]:
base_matches_df = base_matches_df.drop(
    columns=[
        'league_code',
        'home_form_last_3',
        'away_form_last_3',
        'home_goals_half',
        'away_goals_half',
        'result_half',
        'home_corners',
        'away_corners',
        'home_yellow_cards',
        'away_yellow_cards',
        'home_red_cards',
        'away_red_cards',
        'handicap_size'
    ]
)

## Feature engeneering

### Calculating Goals and Expected Goals Difference

Compute the goals and expected goals difference for each match

In [25]:
base_matches_df['goal_diff'] = base_matches_df['home_goals_full'] - base_matches_df['away_goals_full']
base_matches_df['xG_diff'] = base_matches_df['xG_h'] - base_matches_df['xG_a']

### Rolling averages:
Before adding new features, I should create a helping datasets! \
These helping datasets will store the matches in format: **team, opponent, venue(Home/Away)**!

This should be done, so that when calculating rolling features, the teams forms to be gathered from all their recent matches, regarding if they are the home or the away team!So what I will do is that I will take for each team, its home matches and its away matches, calculate their rolling form and then I will merge the helping dataset with the base one.

This way:
- home rolling stats are correct
- away rolling stats are correct
- all previous matches are included
- no leakage
- chronology preserved

First I will get only the home teams:

In [26]:
home_df = base_matches_df[
    [
        "datetime", 
        "h_title", 
        "a_title", 
        "home_goals_full", 
        "away_goals_full", 
        "home_shots", 
        "away_shots", 
        "home_shots_on_target",
        "away_shots_on_target", 
        "xG_h", 
        "xG_a"
    ]
].rename(columns={
    "h_title": "team",
    "a_title": "opponent",
    "home_goals_full": "goals",
    "away_goals_full": "goals_against",
    "home_shots": "shots",
    "away_shots": "shots_against",
    "home_shots_on_target": "sot",
    "away_shots_on_target": "sot_against",
    "xG_h": "xG",
    "xG_a": "xGA"
})

# Calculating goals diff:
home_df["goal_diff"] = (
    home_df["goals"] -
    home_df["goals_against"]
)

home_df["venue"] = "home"

In [27]:
away_df = base_matches_df[
    [
        "datetime",
        "a_title",
        "h_title",
        "away_goals_full",
        "home_goals_full",
        "away_shots",
        "home_shots",
        "away_shots_on_target",
        "home_shots_on_target",
        "xG_a",
        "xG_h"
    ]
].rename(columns={
    "a_title": "team",
    "h_title": "opponent",
    "away_goals_full": "goals",
    "home_goals_full": "goals_against",
    "away_shots": "shots",
    "home_shots": "shots_against",
    "away_shots_on_target": "sot",
    "home_shots_on_target": "sot_against",
    "xG_a": "xG",
    "xG_h": "xGA"
})

# Calculating goal diff:
away_df["goal_diff"] = (
    away_df["goals"] -
    away_df["goals_against"]
)

away_df["venue"] = "away"

Now, as I have rightfully devided the teams as home and away ones, I will concat them:

In [28]:
team_matches = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

In [29]:
# Sorting the matches in chronological order
team_matches = team_matches.sort_values(
    ["team", "datetime", "venue"]
).reset_index(drop=True)

### Calculating rolling averages:

1.Expected goals:
- **home_rolling_xG_last_5, away_rolling_xG_last_5** \
- **home_rolling_xGA_last_5, away_rolling_xGA_last_5**

2.Total Results
- **rolling_goals_last_5**
- **rolling_shots_last_5, rolling_sot_last_5**

3.Form(Points gathred from matches)
- **home_form_last_5, away_form_last_5**

Computing rolling averages for key performance metrics over the last 5 matches for each team

In [30]:
def calculate_rolling_average(data: pd.DataFrame, 
                              group_col: str, 
                              column:str, 
                              window=5) -> pd.Series:
    """
    Calculate the rolling average of a column for each team.

    Parameters:
        data (DataFrame): The input DataFrame
        group_col (str): The column by which to group the data(e.g. home/away team name)
        column (str): The column to calculate the rolling average for
        window (int): The number of games to include in the rolling average

    Returns:
        Series: The rolling average
    """
    
    # This will get the values from the last 5 rows(if window=5).It will not include the current match, because this will cause data leakage!
    # This way it will form a rolling average value of the specific row
    # This will produce some NaN values for the first matches of the teams, just because there is no history data before that, which is logical!
    # The NaNs will be left for now, but later before modeling, they should be filled with either mean values or make them to be handled from the Models! 
    return data.groupby(group_col, observed=False)[column].transform(
        lambda x: x
                .shift(1)
                .rolling(window=window, min_periods=1)
                .mean()
                # .fillna(x.expanding().mean()) # This is made for the rows, which neither does not have historcial data before them or because of another specific problem
    )

In [31]:
rolling_features = [
    "xG",
    "xGA",
    "goals",
    "goals_against",
    "shots",
    "shots_against",
    "sot",
    "sot_against",
    "goal_diff"
]

for feature in rolling_features:
    rolling_col = f"rolling_{feature}_5"
    
    team_matches[rolling_col] = calculate_rolling_average(
        team_matches,
        group_col='team',
        column=feature
    )

It is expected that 31 rows will be with NaN values, because in the dataset there are 31 unique teams, and each of their first matches has not historical data, which produces the NaN values!

So lets see if they are actally 31 rows with NaN values:

In [32]:
team_matches['team'].nunique() == len(team_matches[team_matches.isna().any(axis=1)])

True

Yes they are exactly 31!

##### Now lets again seperate the maches by home and away and prepare them for merge with the base matches dataset:

In [33]:
home_rolling = (
    team_matches[team_matches["venue"] == "home"]
    .rename(columns={
        "team": "h_title",
        "rolling_goals_5": "h_rolling_goals_5",
        "rolling_goals_against_5": "h_rolling_goals_against_5",
        "rolling_shots_5": "h_rolling_shots_5",
        "rolling_shots_against_5": "h_rolling_shots_against_5",
        "rolling_sot_5": "h_rolling_sot_5",
        "rolling_sot_against_5": "h_rolling_sot_against_5",
        "rolling_xG_5": "h_rolling_xG_5",
        "rolling_xGA_5": "h_rolling_xGA_5",
        "rolling_goal_diff_5": "h_rolling_goal_diff_5"
    })
)

home_rolling = home_rolling[
    [
        "datetime",
        "h_title",
        "h_rolling_goals_5",
        "h_rolling_goals_against_5",
        "h_rolling_shots_5",
        "h_rolling_shots_against_5",
        "h_rolling_sot_5",
        "h_rolling_sot_against_5",
        "h_rolling_xG_5",
        "h_rolling_xGA_5",
        "h_rolling_goal_diff_5"
    ]
]

Now the away:

In [34]:
away_rolling = (
    team_matches[team_matches["venue"] == "away"]
    .rename(columns={
        "team": "a_title",
        "rolling_goals_5": "a_rolling_goals_5",
        "rolling_goals_against_5": "a_rolling_goals_against_5",
        "rolling_shots_5": "a_rolling_shots_5",
        "rolling_shots_against_5": "a_rolling_shots_against_5",
        "rolling_sot_5": "a_rolling_sot_5",
        "rolling_sot_against_5": "a_rolling_sot_against_5",
        "rolling_xG_5": "a_rolling_xG_5",
        "rolling_xGA_5": "a_rolling_xGA_5",
        "rolling_goal_diff_5": "a_rolling_goal_diff_5"
    })
)

away_rolling = away_rolling[
    [
        "datetime",
        "a_title",
        "a_rolling_goals_5",
        "a_rolling_goals_against_5",
        "a_rolling_shots_5",
        "a_rolling_shots_against_5",
        "a_rolling_sot_5",
        "a_rolling_sot_against_5",
        "a_rolling_xG_5",
        "a_rolling_xGA_5",
        "a_rolling_goal_diff_5"
    ]
]

Now lets finaly merge the two helping datasets back to the original base matches dataset: 

In [35]:
base_matches_df = base_matches_df.merge(
    home_rolling,
    on=["datetime", "h_title"],
    how="left"
)

base_matches_df = base_matches_df.merge(
    away_rolling,
    on=["datetime", "a_title"],
    how="left"
)

In [36]:
base_matches_df.columns

Index(['season', 'datetime', 'h_title', 'a_title', 'home_elo', 'away_elo',
       'home_form_last_5', 'away_form_last_5', 'home_goals_full',
       'away_goals_full', 'result_full', 'home_shots', 'away_shots',
       'home_shots_on_target', 'away_shots_on_target', 'home_fouls',
       'away_fouls', 'handicap_home', 'handicap_away', 'xG_h', 'xG_a',
       'odds_bet365_home', 'odds_bet365_draw', 'odds_bet365_away', 'goal_diff',
       'xG_diff', 'h_rolling_goals_5', 'h_rolling_goals_against_5',
       'h_rolling_shots_5', 'h_rolling_shots_against_5', 'h_rolling_sot_5',
       'h_rolling_sot_against_5', 'h_rolling_xG_5', 'h_rolling_xGA_5',
       'h_rolling_goal_diff_5', 'a_rolling_goals_5',
       'a_rolling_goals_against_5', 'a_rolling_shots_5',
       'a_rolling_shots_against_5', 'a_rolling_sot_5',
       'a_rolling_sot_against_5', 'a_rolling_xG_5', 'a_rolling_xGA_5',
       'a_rolling_goal_diff_5'],
      dtype='str')

### Calculating **Head-to-Head** Records

Determine the head-to-head performance against each opponent.

To do that I should again split the dataset into home and away matchs, because the calculation of the Head-to_Head metric is calculated different at the different perspectives!

First home team:

In [37]:
home_h2h = base_matches_df[
    [
        "datetime",
        "h_title",
        "a_title",
        "goal_diff",
        "xG_diff"
    ]
].copy()

home_h2h = home_h2h.rename(columns={
    "h_title": "team",
    "a_title": "opponent"
})

# Home perspective:
# positive goal_diff means HOME team performed better
home_h2h["h2h_goal_diff"] = home_h2h["goal_diff"]
home_h2h["h2h_xG_diff"] = home_h2h["xG_diff"]

home_h2h["venue"] = "home"

Away team:

In [38]:
away_h2h = base_matches_df[
    [
        "datetime",
        "h_title",
        "a_title",
        "goal_diff",
        "xG_diff"
    ]
].copy()

away_h2h = away_h2h.rename(columns={
    "a_title": "team",
    "h_title": "opponent"
})

# Away perspective:
# Reversing the sign because: if home team won by +2, away team lost by -2.This is important to be considered
away_h2h["h2h_goal_diff"] = -away_h2h["goal_diff"]
away_h2h["h2h_xG_diff"] = -away_h2h["xG_diff"]

away_h2h["venue"] = "away"

Concating them into one dataset:

In [39]:
h2h_df = pd.concat(
    [home_h2h, away_h2h],
    ignore_index=True
)

In [40]:
h2h_df = h2h_df.sort_values(
    ['team', 'opponent', 'datetime']
).reset_index(drop=True)

Creating the function which will calculate the head to head goal/xG diff rolling averages of the teams.Something which is very important is that this function will adjest more weight over the recent matches, than the past ones.This is something very important because this way, the metris will be based more on the recent team form, which is the right way to compare two teams.It is not logical to compare two teams by their very very old and past forms!

So what the function will do, is that it will simpy define an array of increasing weights which will be applied in the calculation of the averages.And this way for the most recent matches, there will be aplied the more bigger weights and more smaller weights for the oldest ones!  

In [41]:
def weighted_rolling_average(values, window=5) -> pd.Series:
    """
    Calculates weighted rolling average where
    recent matches matter more.

    Parameters:
        values (Series) : the column on which the rolling average to be calculated from
        window (int) : The max amount of past matches to include into the calculation of the rolling average!

    Returns:
        Series: The calculated rolling averages 
    
    Example weights for window=5:
    [1, 2, 3, 4, 5]

    Most recent match receives highest weight.
    """

    results = []

    for i in range(len(values)):

        # Exclude current match to prevent leakage
        historical = values.iloc[max(0, i-window):i]

        # No previous history
        if len(historical) == 0:
            results.append(np.nan)
            continue

        # Increasing weights:
        # older -> lower
        # recent -> higher
        weights = np.arange(1, len(historical) + 1)

        weighted_avg = np.average(
            historical,
            weights=weights
        )

        results.append(weighted_avg)

    return pd.Series(results, index=values.index)

Calculating the rolling averages for goal diff and expected goals diff:

In [42]:
h2h_df["h2h_rolling_goal_diff_5"] = (
    h2h_df.groupby(
        ["team", "opponent"],
        observed=False
    )["h2h_goal_diff"]
    .transform(lambda x: weighted_rolling_average(x, window=5))
)

h2h_df["h2h_rolling_xG_diff_5"] = (
    h2h_df.groupby(
        ["team", "opponent"],
        observed=False
    )["h2h_xG_diff"]
    .transform(lambda x: weighted_rolling_average(x, window=5))
)

Some of the matches does not have history, and because of that the rolling features are with NaN values. \
So I will just fill these values with zeros, in order to remove the NaN:

In [43]:
h2h_df["h2h_rolling_goal_diff_5"] = (
    h2h_df["h2h_rolling_goal_diff_5"]
    .fillna(0)
)

h2h_df["h2h_rolling_xG_diff_5"] = (
    h2h_df["h2h_rolling_xG_diff_5"]
    .fillna(0)
)

Splitting the df into home and away matches, so to be merged with the base matches dataset:

In [44]:
home_h2h_features = (
    h2h_df[h2h_df['venue'] == 'home']
    .rename(columns={
        "team": "h_title",
        "opponent": "a_title",
        "h2h_rolling_goal_diff_5": "h_h2h_rolling_goal_diff_5",
        "h2h_rolling_xG_diff_5": "h_h2h_rolling_xG_diff_5"
    })
)

home_h2h_features = home_h2h_features[
    [
        "datetime",
        "h_title",
        "a_title",
        "h_h2h_rolling_goal_diff_5",
        "h_h2h_rolling_xG_diff_5"
    ]
]

In [45]:
away_h2h_features = (
    h2h_df[h2h_df['venue'] == 'away']
    .rename(columns={
        "team": "a_title",
        "opponent": "h_title",
        "h2h_rolling_goal_diff_5": "a_h2h_rolling_goal_diff_5",
        "h2h_rolling_xG_diff_5": "a_h2h_rolling_xG_diff_5"
    })
)

away_h2h_features = away_h2h_features[
    [
        "datetime",
        "h_title",
        "a_title",
        "a_h2h_rolling_goal_diff_5",
        "a_h2h_rolling_xG_diff_5"
    ]
]

Merge with the original base matches dataset:

In [46]:
base_matches_df = base_matches_df.merge(
    home_h2h_features,
    on=["datetime", "h_title", "a_title"],
    how="left"
)

base_matches_df = base_matches_df.merge(
    away_h2h_features,
    on=["datetime", "h_title", "a_title"],
    how="left"
)

### Elo ratings diff:

Calculating elo ratings diff per match:

In [47]:
base_matches_df['elo_rating_diff'] = base_matches_df['home_elo'] - base_matches_df['away_elo']

### Optimizing the base matches dataset memory usage

In [48]:
base_matches_df = optimize_dataframe_memory(base_matches_df)

Initial Memory Usage: 1.70 MB
Final Memory Usage: 0.62 MB
Memory Reduction: 63.60%

Optimized Data Types:
season                             category
datetime                     datetime64[us]
h_title                            category
a_title                            category
home_elo                            float32
away_elo                            float32
home_form_last_5                       int8
away_form_last_5                       int8
home_goals_full                        int8
away_goals_full                        int8
result_full                        category
home_shots                             int8
away_shots                             int8
home_shots_on_target                   int8
away_shots_on_target                   int8
home_fouls                             int8
away_fouls                             int8
handicap_home                       float32
handicap_away                       float32
xG_h                                float32
xG_a          

---

### Matches shots aggregation

Now I will use the Understat Matches Shots dataset to calculate different shots metrics. \
After the shots aggregation dataset is ready, it will be merged with the Base Matches Dataset and they together will form the **Final Dataset**!

In [49]:
understat_matches_shots_df.columns

Index(['match_id', 'season', 'datetime', 'minute', 'situation', 'X', 'Y',
       'shot_type', 'shot_result', 'last_action', 'xG', 'h_title', 'a_title',
       'player', 'player_id', 'h_a'],
      dtype='str')

In [50]:
shots_df = understat_matches_shots_df.copy()

### Removing columns:

In [51]:
shots_df = shots_df.drop(
    columns=['player', 'player_id']
)

### Adding new features:

#### Defining big chances:
I will start by creating a big chances features, which will simply indicate if the particular shot had a big chance to end up in a goal!Of course, this will be calculated using the expected goals(xG) metric!

To calculate this metric, it should be defined at which value of the expected goal, it could be considered a big chance!

#### Creating the big chance feature

> A shot is considered to be dangerous, when it has more then or equal to 35% probability of ending up in a goal, which means that the **xG**, should be more than or equal then 35%![0][1]

In [52]:
BIG_CHANCE_THRESHOLD = 0.35

In [53]:
shots_df["big_chance"] = (
    shots_df["xG"] >= BIG_CHANCE_THRESHOLD
).astype(int)

### Creating goal flag:

This feature is created so that to be used for measuring **shot conversion**!

In [54]:
shots_df["is_goal"] = (
    shots_df["shot_result"] == "Goal"
).astype(int)

### Splitting the shots into home and away ones:

In order to calculate shot features rightfully I should devide the shots data by home shots(the ones taken from home team) and away shots(the ones taken from the away team), and calculate the shot features for each of the teams!

In [55]:
home_shots = shots_df[
    shots_df["h_a"] == "h"
].copy()

away_shots = shots_df[
    shots_df["h_a"] == "a"
].copy()

### Aggreagting home shot features

First I will aggregate the home shots features:

In [56]:
home_shots['is_box_shot'] = home_shots['X'] >= 0.83
home_shots['is_high_xg'] = home_shots['xG'] >= 0.35
home_shots['open_play_xG'] = home_shots['xG'].where(home_shots['situation'] == 'OpenPlay', 0)
home_shots['set_piece_xG'] = home_shots['xG'].where(home_shots['situation'] != 'OpenPlay', 0)

In [57]:
home_features = (
    home_shots
    .groupby(['match_id', 'season', 'datetime', 'h_title', 'a_title'], observed=True)
    .agg(
        home_big_chances=("big_chance", "sum"),
        home_avg_xG_per_shot=("xG", "mean"),
        home_total_xG=("xG", "sum"),
        home_total_shots=("xG", "size"),
        home_shot_conversion=("is_goal", "mean"),
        home_avg_shot_distance=("X", "mean"),
        home_box_shots=("is_box_shot", "sum"),
        home_high_xg_shots=("is_high_xg", "sum"),
        home_open_play_xG=("open_play_xG", "sum"),
        home_set_piece_xG=("set_piece_xG", "sum")
    )
    .reset_index()
)

In [58]:
len(home_features) # Expecting to have 4180 total matches

4180

Now lets aggregate the away shot features:

In [59]:
away_shots['is_box_shot'] = away_shots['X'] >= 0.83
away_shots['is_high_xg'] = away_shots['xG'] >= 0.35
away_shots['open_play_xG'] = away_shots['xG'].where(away_shots['situation'] == 'OpenPlay', 0)
away_shots['set_piece_xG'] = away_shots['xG'].where(away_shots['situation'] != 'OpenPlay', 0)

In [60]:
away_features = (
    away_shots
    .groupby(['match_id', 'season', 'datetime', 'h_title', 'a_title'], observed=True)
    .agg(
        away_big_chances=("big_chance", "sum"),
        away_avg_xG_per_shot=("xG", "mean"),
        away_total_xG=("xG", "sum"),
        away_total_shots=("xG", "size"),
        away_shot_conversion=("is_goal", "mean"),
        away_avg_shot_distance=("X", "mean"),
        away_box_shots=("is_box_shot", "sum"),
        away_high_xg_shots=("is_high_xg", "sum"),
        away_open_play_xG=("open_play_xG", "sum"),
        away_set_piece_xG=("set_piece_xG", "sum")
    )
    .reset_index()
)

In [61]:
len(away_features) # Expecting to have 4180 total matches

4177

There are three missing matches, which will will be handled later.

### Merging the shot datasets:

As the features of the home and away shots are calculated, the datasets can be merged back together:

In [62]:
shot_features = home_features.merge(
    away_features,
    on=['match_id', 'datetime', 'h_title', 'a_title'],
    how='left',
    suffixes=["", "_a"]
)

In [63]:
shot_features = shot_features.drop(columns=['season_a'])

In [64]:
shot_features.columns

Index(['match_id', 'season', 'datetime', 'h_title', 'a_title',
       'home_big_chances', 'home_avg_xG_per_shot', 'home_total_xG',
       'home_total_shots', 'home_shot_conversion', 'home_avg_shot_distance',
       'home_box_shots', 'home_high_xg_shots', 'home_open_play_xG',
       'home_set_piece_xG', 'away_big_chances', 'away_avg_xG_per_shot',
       'away_total_xG', 'away_total_shots', 'away_shot_conversion',
       'away_avg_shot_distance', 'away_box_shots', 'away_high_xg_shots',
       'away_open_play_xG', 'away_set_piece_xG'],
      dtype='str')

### Creating rolling averages:

Calculating shots rolling averages:
- rolling_home_big_chances_5
- rolling_shot_conversion
- rolling_avg_xG_per_shot_5
- rolling_open_play_xG_5
- rolling_set_piece_xG_5

First I will devide the teams into home and away ones:

In [65]:
home_shots_df = shot_features[
    [
        'datetime', 
        'h_title', 
        'a_title', 
        'home_big_chances',
        'home_avg_xG_per_shot',
        'home_shot_conversion',
        'home_open_play_xG', 
        'home_set_piece_xG'
    ]
].rename(columns={
    "h_title": "team",
    "a_title": "opponent",
    'home_big_chances': 'big_chances',
    'home_avg_xG_per_shot': 'avg_xG_per_shot',
    'home_shot_conversion': 'shot_conversion',
    'home_open_play_xG': 'open_play_xG', 
    'home_set_piece_xG': 'set_piece_xG'
})

home_shots_df["venue"] = "home"

In [66]:
away_shots_df = shot_features[
    [
        'datetime',
        'a_title', 
        'h_title', 
        'away_big_chances',
        'away_avg_xG_per_shot',
        'away_shot_conversion',
        'away_open_play_xG', 
        'away_set_piece_xG'
    ]
].rename(columns={
    "a_title": "team",
    "h_title": "opponent",
    'away_big_chances': 'big_chances',
    'away_avg_xG_per_shot': 'avg_xG_per_shot',
    'away_shot_conversion': 'shot_conversion',
    'away_open_play_xG': 'open_play_xG', 
    'away_set_piece_xG': 'set_piece_xG'
})

away_shots_df["venue"] = "away"

In [67]:
# Merging the datasets
team_shots = pd.concat(
    [home_shots_df, away_shots_df],
    ignore_index=True
)

# Sorting the matches in chronological order
team_shots = team_shots.sort_values(
    ["team", "datetime", "venue"]
).reset_index(drop=True)

#### Calculating shots rolling averages:

In [68]:
shot_rolling_features = [
    'big_chances',
    'avg_xG_per_shot',
    'shot_conversion',
    'open_play_xG', 
    'set_piece_xG'
]

for feature in shot_rolling_features:
    rolling_col = f"rolling_{feature}_5"
    
    team_shots[rolling_col] = calculate_rolling_average(
        team_shots,
        group_col='team',
        column=feature
    )

In [69]:
team_shots.columns

Index(['datetime', 'team', 'opponent', 'big_chances', 'avg_xG_per_shot',
       'shot_conversion', 'open_play_xG', 'set_piece_xG', 'venue',
       'rolling_big_chances_5', 'rolling_avg_xG_per_shot_5',
       'rolling_shot_conversion_5', 'rolling_open_play_xG_5',
       'rolling_set_piece_xG_5'],
      dtype='str')

Now as we calculated the rolling averages for each of the teams(home/away), we should seperate them again:

In [70]:
home_shot_rolling = (
    team_shots[team_shots["venue"] == "home"]
    .rename(columns={
        "team": "h_title",
        'rolling_big_chances_5': 'h_rolling_big_chances_5', 
        'rolling_avg_xG_per_shot_5': 'h_rolling_avg_xG_per_shot_5',
        'rolling_shot_conversion_5': 'h_rolling_shot_conversion_5', 
        'rolling_open_play_xG_5': 'h_rolling_open_play_xG_5',
        'rolling_set_piece_xG_5': 'h_rolling_set_piece_xG_5'
    })
)

home_shot_rolling = home_shot_rolling[
    [
        "datetime",
        "h_title",
        'h_rolling_big_chances_5', 
        'h_rolling_avg_xG_per_shot_5',
        'h_rolling_shot_conversion_5', 
        'h_rolling_open_play_xG_5',
        'h_rolling_set_piece_xG_5'
    ]
]

Now for the away:

In [71]:
away_shot_rolling = (
    team_shots[team_shots["venue"] == "away"]
    .rename(columns={
        "team": "a_title",
        'rolling_big_chances_5': 'a_rolling_big_chances_5', 
        'rolling_avg_xG_per_shot_5': 'a_rolling_avg_xG_per_shot_5',
        'rolling_shot_conversion_5': 'a_rolling_shot_conversion_5', 
        'rolling_open_play_xG_5': 'a_rolling_open_play_xG_5',
        'rolling_set_piece_xG_5': 'a_rolling_set_piece_xG_5'
    })
)

away_shot_rolling = away_shot_rolling[
    [
        "datetime",
        "a_title",
        'a_rolling_big_chances_5', 
        'a_rolling_avg_xG_per_shot_5',
        'a_rolling_shot_conversion_5', 
        'a_rolling_open_play_xG_5',
        'a_rolling_set_piece_xG_5'
    ]
]

In [72]:
shot_features = shot_features.merge(
    home_shot_rolling,
    on=["datetime", "h_title"],
    how="left"
)

shot_features = shot_features.merge(
    away_shot_rolling,
    on=["datetime", "a_title"],
    how="left"
)

In [73]:
shot_features.columns

Index(['match_id', 'season', 'datetime', 'h_title', 'a_title',
       'home_big_chances', 'home_avg_xG_per_shot', 'home_total_xG',
       'home_total_shots', 'home_shot_conversion', 'home_avg_shot_distance',
       'home_box_shots', 'home_high_xg_shots', 'home_open_play_xG',
       'home_set_piece_xG', 'away_big_chances', 'away_avg_xG_per_shot',
       'away_total_xG', 'away_total_shots', 'away_shot_conversion',
       'away_avg_shot_distance', 'away_box_shots', 'away_high_xg_shots',
       'away_open_play_xG', 'away_set_piece_xG', 'h_rolling_big_chances_5',
       'h_rolling_avg_xG_per_shot_5', 'h_rolling_shot_conversion_5',
       'h_rolling_open_play_xG_5', 'h_rolling_set_piece_xG_5',
       'a_rolling_big_chances_5', 'a_rolling_avg_xG_per_shot_5',
       'a_rolling_shot_conversion_5', 'a_rolling_open_play_xG_5',
       'a_rolling_set_piece_xG_5'],
      dtype='str')

#### Now the dataset contains rolling shots metrcis!

Actually, the reason why the rolling features are so important, is because they provide the teams performances before the actual match, without including any current match stats, because otherwise, without these rolling features, the matches only containts stats from the matches themselves, which cannot be used in predictions, because it is expected that the matches stats are unknown yet, so the things that should be used are precisely the rolling features, because they provide performance before the actual matches!!!

### Removing features from the shots dataset

Some features are useless and should be removed:

In [74]:
shot_features = shot_features.drop(
    columns=[
        'match_id',
        'home_total_xG',
        'away_total_xG',
        'home_total_shots',
        'away_total_shots'
    ]
)

In [75]:
shot_features.columns

Index(['season', 'datetime', 'h_title', 'a_title', 'home_big_chances',
       'home_avg_xG_per_shot', 'home_shot_conversion',
       'home_avg_shot_distance', 'home_box_shots', 'home_high_xg_shots',
       'home_open_play_xG', 'home_set_piece_xG', 'away_big_chances',
       'away_avg_xG_per_shot', 'away_shot_conversion',
       'away_avg_shot_distance', 'away_box_shots', 'away_high_xg_shots',
       'away_open_play_xG', 'away_set_piece_xG', 'h_rolling_big_chances_5',
       'h_rolling_avg_xG_per_shot_5', 'h_rolling_shot_conversion_5',
       'h_rolling_open_play_xG_5', 'h_rolling_set_piece_xG_5',
       'a_rolling_big_chances_5', 'a_rolling_avg_xG_per_shot_5',
       'a_rolling_shot_conversion_5', 'a_rolling_open_play_xG_5',
       'a_rolling_set_piece_xG_5'],
      dtype='str')

### Fixing data types

Some of the columns are with unappropriate data types, so they should be fixed:

In [76]:
shot_features = validate_and_cast_dataframe_dtypes(shot_features)

Ok now every column is with appropriate data type!

### Optimizing dataset memory usage:

Most of the columns can be casted to more small-sized types, which will reduce a lot of memory!

In [77]:
shot_features = optimize_dataframe_memory(shot_features)

Initial Memory Usage: 0.76 MB
Final Memory Usage: 0.48 MB
Memory Reduction: 36.25%

Optimized Data Types:
season                               category
datetime                       datetime64[us]
h_title                              category
a_title                              category
home_big_chances                         int8
home_avg_xG_per_shot                  float32
home_shot_conversion                  float32
home_avg_shot_distance                float32
home_box_shots                           int8
home_high_xg_shots                       int8
home_open_play_xG                     float32
home_set_piece_xG                     float32
away_big_chances                        Int64
away_avg_xG_per_shot                  float32
away_shot_conversion                  float32
away_avg_shot_distance                float32
away_box_shots                          Int64
away_high_xg_shots                      Int64
away_open_play_xG                     float32
away_set_piece_xG   

### Ordering the shot features dataset

In [78]:
shot_features = shot_features.sort_values('datetime').reset_index(drop=True)

## Creating the final dataset:

Now as all of the features have been created, and all of the components are cleaned, it's time to **merge the Base Matches dataset and the Shot Features dataset**!

But before merging I should be ensure that there are no duplicated matches across the datasets:

In [79]:
shot_features.duplicated(subset=['datetime', 'h_title', 'a_title']).sum()

np.int64(0)

In [80]:
base_matches_df.duplicated(subset=['datetime', 'h_title', 'a_title']).sum()

np.int64(0)

#### Another thing which is almost certain, is that, as the first **understat matches dataset** was with wrong matches dates(one day ahead of the real date), this shot fetaures dataset will also be with invalid dates, because it is created from the **Understat matches shots** dataset!

### Resolving the **ShotFeatures** dataset matches dates:

In [81]:
base_matches_df = base_matches_df.copy()
shot_features = shot_features.copy()

base_matches_df["base_idx"] = base_matches_df.index
shot_features["shot_idx"] = shot_features.index

In [82]:
test_df = base_matches_df.merge(
    shot_features,
    on=[
        "season",
        "datetime",
        "h_title",
        "a_title"
    ],
    how="outer",
    indicator=True
)

In [83]:
test_df.shape

(4249, 78)

Yes they are certainly more than they should be, so this dataset should also be fixed as the first Understat matches one.

Getting the ones which differ:

In [84]:
left_only = test_df[test_df["_merge"] == "left_only"].copy()
right_only = test_df[test_df["_merge"] == "right_only"].copy()

Now lets apply the **resolve_date_shift_matches** function, which will fix the problem:

In [85]:
resolved_matches = resolve_date_shift_matches(
    left_df=left_only,
    right_df=right_only,
    use_index=True,
    left_index_col='base_idx',
    right_index_col='shot_idx',
    max_days=1
)

In [86]:
resolved_matches

,left_index,right_index
0,384.0,384.0
1,387.0,388.0
2,386.0,387.0
3,392.0,396.0
4,393.0,394.0
...,...,...
64,1170.0,1171.0
65,1224.0,1229.0
66,1242.0,1244.0
67,3020.0,3022.0


Now lets use the resolved matches that we have craeted and fix the **ShotFeature dataset**:

In [87]:
for _, row in resolved_matches.iterrows():

    left_idx = row["left_index"]
    right_idx = row["right_index"]

    corrected_date = base_matches_df.loc[left_idx, "datetime"]

    shot_features.loc[right_idx, "datetime"] = corrected_date

Now lets remove the helping index columns:

In [88]:
base_matches_df = base_matches_df.drop(columns=['base_idx'])
shot_features = shot_features.drop(columns=['shot_idx'])

In [89]:
final_df = base_matches_df.merge(
    shot_features,
    on=[
        "season",
        "datetime",
        "h_title",
        "a_title"
    ],
    how="inner",
)

In [90]:
final_df.shape

(4180, 75)

Ahh, now everything seems to be perfect:

In [91]:
final_df.columns

Index(['season', 'datetime', 'h_title', 'a_title', 'home_elo', 'away_elo',
       'home_form_last_5', 'away_form_last_5', 'home_goals_full',
       'away_goals_full', 'result_full', 'home_shots', 'away_shots',
       'home_shots_on_target', 'away_shots_on_target', 'home_fouls',
       'away_fouls', 'handicap_home', 'handicap_away', 'xG_h', 'xG_a',
       'odds_bet365_home', 'odds_bet365_draw', 'odds_bet365_away', 'goal_diff',
       'xG_diff', 'h_rolling_goals_5', 'h_rolling_goals_against_5',
       'h_rolling_shots_5', 'h_rolling_shots_against_5', 'h_rolling_sot_5',
       'h_rolling_sot_against_5', 'h_rolling_xG_5', 'h_rolling_xGA_5',
       'h_rolling_goal_diff_5', 'a_rolling_goals_5',
       'a_rolling_goals_against_5', 'a_rolling_shots_5',
       'a_rolling_shots_against_5', 'a_rolling_sot_5',
       'a_rolling_sot_against_5', 'a_rolling_xG_5', 'a_rolling_xGA_5',
       'a_rolling_goal_diff_5', 'h_h2h_rolling_goal_diff_5',
       'h_h2h_rolling_xG_diff_5', 'a_h2h_rolling_goal_diff

## Handling missing values:

The last problem which should be handled are the missing values.Most of the values are missing because of the lack of history data from the first matches of the teams, and this is logical.But I cannot leave so many missing values!

So lets first explain why these values are missing:

#### **1.Rolling features**:
Most of the missing values comes from the rolling features metrics, which is logical, because some of the matches simply does not have history data before them.

#### **2.Away shot features**:
There are three away matches, which were taken from the shot features dataset, which were missing.This could have happended because of a incompletness in the shot features dataset, or some problem while collecting the data.This is not very serious because these are only three matches!

So based on the missing values columns, the different features groups should be handled differently.

### Missing rolling features:
All of the missing rolling features will be filled with zeros.
Because:
- no history = no prior information
- zero represents "no previous evidence"

Especially for:
- rolling goals
- rolling xG
- rolling shots
- rolling H2H

In [92]:
rolling_cols = [
    col for col in final_df.columns
    if "rolling" in col
]

final_df[rolling_cols] = (
    final_df[rolling_cols]
    .fillna(0)
)

### Missing Shot Aggregation

Here the different metrics need different filling logic.

The count features, such as the **away_big_chances, away_box_shots, away_high_xg_shots** can be freely filled with zeros:

In [93]:
count_cols = [
    "away_big_chances",
    "away_box_shots",
    "away_high_xg_shots"
]

final_df[count_cols] = (
    final_df[count_cols]
    .fillna(0)
)

And for the **rate / average features** such as the **away_avg_xG_per_shot, away_shot_conversion, away_avg_shot_distance, away_open_play_xG, away_set_piece_xG**, a median value would be more appropriate.

In [94]:
median_cols = [
    "away_avg_xG_per_shot",
    "away_shot_conversion",
    "away_avg_shot_distance",
    "away_open_play_xG",
    "away_set_piece_xG"
]

for col in median_cols:
    final_df[col] = (
        final_df[col]
        .fillna(final_df[col].median())
    )

In [95]:
all(final_df.notna().all())

True

I cannot say that these approaches for dealing with missing values are extrmely right or wrong, professiona or not, but for what I have as a data, and due to the fact that I don't have the history data, required to fill these columns, I think that this is more or less a solid way to hadle these missing values.

However, lets proceed by optimizing this final dataset and saving it into the processed data folder:

## Optimizing the final dataset:

Reducing the memory usage of the dataset:

In [96]:
final_df = optimize_dataframe_memory(final_df)

Initial Memory Usage: 1.06 MB
Final Memory Usage: 0.96 MB
Memory Reduction: 9.03%

Optimized Data Types:
season                               category
datetime                       datetime64[us]
h_title                              category
a_title                              category
home_elo                              float32
                                    ...      
a_rolling_big_chances_5               float32
a_rolling_avg_xG_per_shot_5           float32
a_rolling_shot_conversion_5           float32
a_rolling_open_play_xG_5              float32
a_rolling_set_piece_xG_5              float32
Length: 75, dtype: object


In [97]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4180 entries, 0 to 4179
Data columns (total 75 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   season                       4180 non-null   category      
 1   datetime                     4180 non-null   datetime64[us]
 2   h_title                      4180 non-null   category      
 3   a_title                      4180 non-null   category      
 4   home_elo                     4180 non-null   float32       
 5   away_elo                     4180 non-null   float32       
 6   home_form_last_5             4180 non-null   int8          
 7   away_form_last_5             4180 non-null   int8          
 8   home_goals_full              4180 non-null   int8          
 9   away_goals_full              4180 non-null   int8          
 10  result_full                  4180 non-null   category      
 11  home_shots                   4180 non-null   int8     

## Saving the final dataset

Saving the final dataset into the processed data folder!

In [98]:
# force working directory to project root
PROJECT_ROOT = Path().resolve().parent.parent

In [99]:
processed_final_dataset_path = PROJECT_ROOT / PROCESSED_DATA_PATH / 'processed_final_dataset.parquet'

save_data(data=final_df, file_path=processed_final_dataset_path)

The file has already been created and it contains the exact data as the original dataset!
